# NLP Review Categorization for Kaggle
This notebook runs the Zero-Shot NLP model to categorize customer reviews and saves the results to `/kaggle/working/nlp_root_causes.csv`.

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
sys.modules['tensorflow'] = None

import pandas as pd
from transformers import pipeline
from tqdm.notebook import tqdm
import warnings
import torch

warnings.filterwarnings('ignore')
tqdm.pandas()

# Check for GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Load dataset from Kaggle input path
input_path = '/kaggle/input/datasets/devyanshagarwal/review/olist_order_reviews_dataset.csv'
df_reviews = pd.read_csv(input_path)

bad_reviews = df_reviews[(df_reviews['review_score'] < 4) & (df_reviews['review_comment_message'].notna())].copy()
print(f"Total Negative Reviews with Text: {len(bad_reviews)}")

# Initialize classifier
# Using float32 to avoid 'CUDA error: no kernel image is available for execution on the device' which can happen with fp16 on older architectures.
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device, torch_dtype=torch.float32)

candidate_labels_pt = [
    "Atraso na entrega ou problema logístico",
    "Produto com defeito ou qualidade ruim",  
    "Produto errado entregue",                
    "Problema com o atendimento ou reembolso" 
]

translation_map = {
    "Atraso na entrega ou problema logístico": "Logistics & Delivery Delay",
    "Produto com defeito ou qualidade ruim": "Defective / Bad Quality Product",
    "Produto errado entregue": "Wrong Product Delivered",
    "Problema com o atendimento ou reembolso": "Customer Service / Refund Issue"
}

def categorize_review(text):
    try:
        res = classifier(str(text), candidate_labels_pt)
        return translation_map[res['labels'][0]]
    except Exception as e:
        return "Classification Error"

demo_batch = bad_reviews.copy()
print("Categorizing reviews... This will take a few minutes on GPU.")
demo_batch['Root_Cause_English'] = demo_batch['review_comment_message'].progress_apply(categorize_review)

# Save output to Kaggle working directory
output_path = '/kaggle/working/nlp_root_causes.csv'
demo_batch[['review_score', 'review_comment_message', 'Root_Cause_English']].to_csv(output_path, index=True)
print(f"Successfully saved {len(demo_batch)} NLP classifications to {output_path}!")
